# Тест multilingual-e5-base vs ru-en-RoSBERTa

Сравниваем два bi-encoder'а на одних и тех же 50k постах:
- `ai-forever/ru-en-RoSBERTa` (384 dims, текущий)
- `intfloat/multilingual-e5-base` (768 dims, из коробки)

Отдельная LanceDB база, 3 тестовых запроса.

In [ ]:
# !pip install lancedb sentence-transformers

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json, random, time
import numpy as np
import torch
import lancedb
import pyarrow as pa
from tqdm.auto import tqdm

import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ==================== НАСТРОЙКИ ====================
POSTS_FILE = "data/posts/ajtkulov/selected/selected500k_cleaned.jsonl"
MAX_POSTS = 50_000
BATCH_SIZE = 256
RANDOM_SEED = 42

# Тестовые запросы — товарные описания
TEST_QUERIES = [
    "Кроссовки мужские для бега Nike Air Max, дышащий верх, амортизирующая подошва, чёрный цвет, размеры 40-45",
    "Увлажняющий крем для лица с гиалуроновой кислотой и витамином C, для сухой и нормальной кожи, 50 мл",
    "Набор LEGO Technic Porsche 911 GT3 RS, 2704 детали, для детей от 16 лет, коллекционная модель",
]

## 1. Загрузка постов

In [ ]:
all_posts = []
with open(POSTS_FILE, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Чтение"):
        obj = json.loads(line)
        if obj.get('text', '').strip():
            cat = obj.get('category', '')
            if '|||' in cat:
                obj['category'] = cat.split('|||')[0].strip()
            all_posts.append(obj)

rng = random.Random(RANDOM_SEED)
posts = rng.sample(all_posts, MAX_POSTS)
print(f"Выборка: {len(posts):,} постов")

## 2. Загрузка моделей

In [ ]:
# Текущий дообученный bi-encoder
model_rosbert = SentenceTransformer("models/final/bi-encoder", device=device)
print(f"RoSBERTa (fine-tuned): dim={model_rosbert.get_sentence_embedding_dimension()}")

# E5-base из коробки
model_e5 = SentenceTransformer("intfloat/multilingual-e5-base", device=device)
print(f"E5-base (из коробки):  dim={model_e5.get_sentence_embedding_dimension()}")

## 3. Векторизация и создание двух баз

In [ ]:
texts = [p['text'] for p in posts]

print("Кодируем RoSBERTa...")
t0 = time.time()
emb_rosbert = model_rosbert.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=BATCH_SIZE)
t_rosbert = time.time() - t0
print(f"  {t_rosbert:.0f}с ({len(texts)/t_rosbert:.0f} п/с)")

# E5 требует префикс "query: " для запросов и "passage: " для документов
print("\nКодируем E5-base...")
e5_texts = ["passage: " + t for t in texts]
t0 = time.time()
emb_e5 = model_e5.encode(e5_texts, normalize_embeddings=True, show_progress_bar=True, batch_size=BATCH_SIZE)
t_e5 = time.time() - t0
print(f"  {t_e5:.0f}с ({len(texts)/t_e5:.0f} п/с)")

print(f"\nРазница по скорости: E5 в {t_e5/t_rosbert:.1f}x медленнее")

In [ ]:
import shutil, os

# Чистим старые тестовые базы
for path in ["./lancedb_test_rosbert", "./lancedb_test_e5"]:
    if os.path.exists(path):
        shutil.rmtree(path)

def build_lancedb(db_path, embeddings, dim):
    db = lancedb.connect(db_path)
    schema = pa.schema([
        pa.field("vector", pa.list_(pa.float32(), dim)),
        pa.field("text", pa.utf8()),
        pa.field("channel", pa.utf8()),
        pa.field("category", pa.utf8()),
    ])
    table = db.create_table("posts", schema=schema)
    
    # Пишем батчами
    BS = 5000
    for i in tqdm(range(0, len(posts), BS), desc=db_path):
        batch = posts[i:i+BS]
        embs = embeddings[i:i+BS]
        records = [{
            "vector": embs[j].tolist(),
            "text": batch[j]['text'],
            "channel": batch[j]['channel'],
            "category": batch[j].get('category', ''),
        } for j in range(len(batch))]
        table.add(records)
    
    table.create_fts_index("text", replace=True)
    return db, table

print("Строим базу RoSBERTa...")
db_ros, tbl_ros = build_lancedb("./lancedb_test_rosbert", emb_rosbert, 384)

print("Строим базу E5...")
db_e5, tbl_e5 = build_lancedb("./lancedb_test_e5", emb_e5, 768)

print("Готово!")

## 4. Сравнение результатов поиска

In [ ]:
TOP_K = 10

for q_idx, query in enumerate(TEST_QUERIES, 1):
    print(f"\n{'='*90}")
    print(f"ЗАПРОС {q_idx}: {query[:80]}...")
    print(f"{'='*90}")
    
    # RoSBERTa
    qvec_ros = model_rosbert.encode([query], normalize_embeddings=True)[0].tolist()
    results_ros = tbl_ros.search(qvec_ros, query_type="vector").limit(TOP_K).select(["text", "channel", "category"]).to_list()
    
    # E5 (запрос с префиксом "query: ")
    qvec_e5 = model_e5.encode(["query: " + query], normalize_embeddings=True)[0].tolist()
    results_e5 = tbl_e5.search(qvec_e5, query_type="vector").limit(TOP_K).select(["text", "channel", "category"]).to_list()
    
    # BM25 (одинаковый для обеих баз — тексты те же)
    results_bm25 = tbl_ros.search(query, query_type="fts").limit(TOP_K).select(["text", "channel", "category"]).to_list()
    
    for mode_name, results in [("BM25", results_bm25), ("RoSBERTa (384d, fine-tuned)", results_ros), ("E5-base (768d, из коробки)", results_e5)]:
        print(f"\n  --- {mode_name} ---")
        for i, r in enumerate(results[:5], 1):
            score = r.get('_distance', r.get('_score', r.get('_relevance_score', '?')))
            print(f"  {i}. @{r['channel']} [{r['category']}] score={score:.4f}" if isinstance(score, float) else f"  {i}. @{r['channel']} [{r['category']}] score={score}")
            print(f"     {r['text'][:130]}")

## 5. Итоги

Сравни результаты глазами:
- BM25 — бейзлайн, ищет по словам
- RoSBERTa — твой текущий дообученный bi-encoder (384d)
- E5-base — из коробки, без дообучения (768d)

Если E5 уже из коробки лучше, после дообучения на твоём серебряном датасете будет ещё лучше.